# Final Open-Access Statistical Analysis

In [ ]:
# Core imports used by all analyses
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nolds
import spm1d
import statsmodels.formula.api as smf

from scipy import stats
from scipy.interpolate import interp1d
from statsmodels.stats.multitest import multipletests
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

# All generated Excel files are written here.
OUTPUT_DIR = Path("Statistical_Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
EXPECTED_REPETITIONS = 2


def _fmt_p(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "< .001"
    return f"= {p:.3f}".replace("0.", ".")


def _mean_sd_ci(values, decimals=3):
    values = pd.Series(values, dtype=float).dropna()
    n = len(values)
    if n < 2:
        return "NA"
    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    lo, hi = mean - tcrit * se, mean + tcrit * se
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f}\n[{lo:.{decimals}f}, {hi:.{decimals}f}]"


def _fit_mixedlm(formula, data):
    model = smf.mixedlm(
        formula=formula,
        data=data,
        groups=data["_Subject"],
        re_formula="1",
    )
    errors = []
    for method, maxiter in (("lbfgs", 3000), ("powell", 5000), ("cg", 5000)):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = model.fit(reml=False, method=method, maxiter=maxiter, disp=False)
            if np.isfinite(fit.llf):
                return fit, method, " | ".join(errors)
        except Exception as exc:
            errors.append(f"{method}: {exc}")
    return None, "NA", " | ".join(errors)


def _random_intercept_variance(fit):
    try:
        cov_re = np.asarray(fit.cov_re, dtype=float)
        return float(cov_re[0, 0])
    except Exception:
        return np.nan


def _lmm_test(model_data, condition_order):
    reference = condition_order[0]
    full_formula = f"_Y ~ C(_Condition, Treatment(reference='{reference}'))"
    null_formula = "_Y ~ 1"
    full, opt_full, err_full = _fit_mixedlm(full_formula, model_data)
    null, opt_null, err_null = _fit_mixedlm(null_formula, model_data)

    if full is None or null is None:
        return {
            "full": full, "null": null, "optimizer": f"{opt_full}/{opt_null}",
            "fit_errors": f"{err_full} | {err_null}", "chi2": np.nan, "df": np.nan,
            "p": np.nan, "converged": False, "ri_var": np.nan, "resid_var": np.nan,
            "icc": np.nan, "singular": True,
        }

    chi2 = max(0.0, 2.0 * (float(full.llf) - float(null.llf)))
    df = int(len(full.fe_params) - len(null.fe_params))
    p = float(stats.chi2.sf(chi2, df)) if df > 0 else np.nan
    ri_var = _random_intercept_variance(full)
    resid_var = float(full.scale) if np.isfinite(full.scale) else np.nan
    denom = ri_var + resid_var
    icc = ri_var / denom if np.isfinite(denom) and denom > 0 else np.nan
    singular = bool((not np.isfinite(ri_var)) or ri_var < 1e-8 or (np.isfinite(icc) and icc < 1e-6))

    return {
        "full": full, "null": null, "optimizer": f"{opt_full}/{opt_null}",
        "fit_errors": f"{err_full} | {err_null}", "chi2": chi2, "df": df, "p": p,
        "converged": bool(getattr(full, "converged", False) and getattr(null, "converged", False)),
        "ri_var": ri_var, "resid_var": resid_var, "icc": icc, "singular": singular,
    }


def _diagnostics(full_fit, model_data, condition_order):
    if full_fit is None:
        return {"Shapiro-Wilk W": np.nan, "Shapiro-Wilk p": np.nan,
                "Brown-Forsythe Statistic": np.nan, "Brown-Forsythe p": np.nan,
                "Residual Skewness": np.nan, "Residual Excess Kurtosis": np.nan}
    residuals = np.asarray(full_fit.resid, dtype=float)
    finite = np.isfinite(residuals)
    residuals = residuals[finite]
    if 3 <= len(residuals) <= 5000:
        sw_w, sw_p = stats.shapiro(residuals)
    else:
        sw_w, sw_p = np.nan, np.nan
    groups = []
    full_resid = pd.Series(np.asarray(full_fit.resid, dtype=float), index=model_data.index)
    for level in condition_order:
        vals = full_resid.loc[model_data.index[model_data["_Condition"] == level]].dropna().to_numpy(dtype=float)
        if len(vals) >= 2:
            groups.append(vals)
    if len(groups) == len(condition_order):
        bf_stat, bf_p = stats.levene(*groups, center="median")
    else:
        bf_stat, bf_p = np.nan, np.nan
    return {
        "Shapiro-Wilk W": float(sw_w) if np.isfinite(sw_w) else np.nan,
        "Shapiro-Wilk p": float(sw_p) if np.isfinite(sw_p) else np.nan,
        "Brown-Forsythe Statistic": float(bf_stat) if np.isfinite(bf_stat) else np.nan,
        "Brown-Forsythe p": float(bf_p) if np.isfinite(bf_p) else np.nan,
        "Residual Skewness": float(stats.skew(residuals, bias=False)) if len(residuals) >= 3 else np.nan,
        "Residual Excess Kurtosis": float(stats.kurtosis(residuals, fisher=True, bias=False)) if len(residuals) >= 4 else np.nan,
    }


def _participant_condition_means(model_data, condition_order, expected_repetitions=2):
    pc = (model_data.groupby(["_Metric", "_Subject", "_Condition"], observed=True, as_index=False)
          .agg(Valid_Trials=("_Y", "count"), Value=("_Y", "mean")))
    pc["Complete_Two_Trial_Cell"] = pc["Valid_Trials"].eq(expected_repetitions)
    pc.loc[~pc["Complete_Two_Trial_Cell"], "Value"] = np.nan
    pc["_Condition"] = pd.Categorical(pc["_Condition"], categories=condition_order, ordered=True)
    return pc


def _sensitivity_test(pc_metric, condition_order):
    wide = pc_metric.pivot(index="_Subject", columns="_Condition", values="Value").reindex(columns=condition_order).dropna()
    n = len(wide)
    if n < 3:
        return {"Sensitivity Test": "Insufficient complete participant cells", "Sensitivity Statistic": np.nan,
                "Sensitivity df": np.nan, "Sensitivity p": np.nan, "Sensitivity N": n}
    if len(condition_order) == 2:
        a, b = wide.iloc[:, 0].to_numpy(float), wide.iloc[:, 1].to_numpy(float)
        diff = b - a
        if np.allclose(diff, 0):
            stat, p = 0.0, 1.0
        else:
            try:
                stat, p = stats.wilcoxon(b, a, alternative="two-sided", zero_method="wilcox")
            except ValueError:
                stat, p = np.nan, np.nan
        return {"Sensitivity Test": "Wilcoxon signed-rank", "Sensitivity Statistic": float(stat),
                "Sensitivity df": np.nan, "Sensitivity p": float(p), "Sensitivity N": n}
    arrays = [wide[level].to_numpy(float) for level in condition_order]
    stat, p = stats.friedmanchisquare(*arrays)
    return {"Sensitivity Test": "Friedman", "Sensitivity Statistic": float(stat),
            "Sensitivity df": len(condition_order) - 1, "Sensitivity p": float(p), "Sensitivity N": n}


def _extract_two_level_effect(full_fit, reference, comparison):
    if full_fit is None:
        return {"beta": np.nan, "SE": np.nan, "z": np.nan, "CI Low": np.nan, "CI High": np.nan}
    target = None
    for name in full_fit.fe_params.index:
        if name != "Intercept" and f"T.{comparison}]" in name:
            target = name
            break
    if target is None:
        return {"beta": np.nan, "SE": np.nan, "z": np.nan, "CI Low": np.nan, "CI High": np.nan}
    beta = float(full_fit.fe_params[target])
    se = float(full_fit.bse_fe[target])
    z = beta / se if se > 0 else np.nan
    return {"beta": beta, "SE": se, "z": z, "CI Low": beta - 1.96 * se, "CI High": beta + 1.96 * se}


def _pairwise_lrt(metric_data, level1, level2):
    pair = metric_data[metric_data["_Condition"].isin([level1, level2])].copy()
    pair["_Condition"] = pd.Categorical(pair["_Condition"], categories=[level1, level2], ordered=True)
    test = _lmm_test(pair, [level1, level2])
    effect = _extract_two_level_effect(test["full"], level1, level2)
    return {**test, **effect}


def run_lmm_family(
    long_data,
    metric_col,
    value_col,
    subject_col,
    condition_col,
    condition_order,
    output_filename,
    expected_repetitions=EXPECTED_REPETITIONS,
    decimals=3,
):
    required = [metric_col, value_col, subject_col, condition_col]
    missing = [c for c in required if c not in long_data.columns]
    if missing:
        raise ValueError(f"Missing columns for LMM analysis: {missing}")

    d = long_data[required].copy()
    d.columns = ["_Metric", "_Y", "_Subject", "_Condition"]
    d["_Y"] = pd.to_numeric(d["_Y"], errors="coerce")
    d = d.dropna(subset=["_Metric", "_Y", "_Subject", "_Condition"])
    d = d[d["_Condition"].isin(condition_order)].copy()
    d["_Condition"] = pd.Categorical(d["_Condition"], categories=condition_order, ordered=True)

    pc = _participant_condition_means(d, condition_order, expected_repetitions)
    rows, diag_rows, sens_rows, posthoc_rows = [], [], [], []

    for metric in pd.unique(d["_Metric"]):
        md = d[d["_Metric"] == metric].copy()
        if md["_Subject"].nunique() < 2 or len(md) < 4:
            continue
        test = _lmm_test(md, condition_order)
        diag = _diagnostics(test["full"], md, condition_order)
        pc_m = pc[pc["_Metric"] == metric].copy()
        descriptives = {}
        for level in condition_order:
            vals = pc_m.loc[pc_m["_Condition"] == level, "Value"].dropna()
            descriptives[level] = _mean_sd_ci(vals, decimals=decimals)

        assumption_violation = (
            (np.isfinite(diag["Shapiro-Wilk p"]) and diag["Shapiro-Wilk p"] < ALPHA)
            or (np.isfinite(diag["Brown-Forsythe p"]) and diag["Brown-Forsythe p"] < ALPHA)
            or test["singular"] or (not test["converged"])
        )
        sens = _sensitivity_test(pc_m, condition_order) if assumption_violation else {
            "Sensitivity Test": "Not required", "Sensitivity Statistic": np.nan,
            "Sensitivity df": np.nan, "Sensitivity p": np.nan,
            "Sensitivity N": int(pc_m.pivot(index="_Subject", columns="_Condition", values="Value").dropna().shape[0]),
        }

        row = {
            "Metric": metric,
            "Participants": int(md["_Subject"].nunique()),
            "Trial-level Observations": int(len(md)),
            **{level: descriptives[level] for level in condition_order},
            "Primary Model": "Outcome ~ Condition + (1 | Participant)",
            "Primary Test": "Likelihood-ratio test",
            "LR chi-square": test["chi2"],
            "df": test["df"],
            "Raw p": test["p"],
            "Converged": test["converged"],
            "Singular Random Effect": test["singular"],
            "Random Intercept Variance": test["ri_var"],
            "Residual Variance": test["resid_var"],
            "ICC": test["icc"],
            "Optimizer": test["optimizer"],
        }
        if len(condition_order) == 2:
            eff = _extract_two_level_effect(test["full"], condition_order[0], condition_order[1])
            row.update({"beta": eff["beta"], "SE": eff["SE"], "Wald z": eff["z"],
                        "95% CI Low": eff["CI Low"], "95% CI High": eff["CI High"]})
        rows.append(row)
        diag_rows.append({"Metric": metric, **diag, "Assumption/Singularity Flag": assumption_violation})
        sens_rows.append({"Metric": metric, **sens})

    results = pd.DataFrame(rows)
    diagnostics = pd.DataFrame(diag_rows)
    sensitivity = pd.DataFrame(sens_rows)
    if results.empty:
        raise ValueError("No usable mixed-effects models were fitted.")

    results["FDR q"] = np.nan
    results["Significant After FDR"] = False
    valid = results["Raw p"].notna() & np.isfinite(results["Raw p"].to_numpy(float))
    if valid.any():
        reject, q, _, _ = multipletests(results.loc[valid, "Raw p"], alpha=ALPHA, method="fdr_bh")
        results.loc[valid, "FDR q"] = q
        results.loc[valid, "Significant After FDR"] = reject

    if not sensitivity.empty:
        sensitivity["Sensitivity FDR q"] = np.nan
        sv = sensitivity["Sensitivity p"].notna() & np.isfinite(sensitivity["Sensitivity p"].fillna(np.nan).to_numpy(float))
        if sv.any():
            _, sq, _, _ = multipletests(sensitivity.loc[sv, "Sensitivity p"], alpha=ALPHA, method="fdr_bh")
            sensitivity.loc[sv, "Sensitivity FDR q"] = sq

    if len(condition_order) > 2:
        for metric in results.loc[results["Significant After FDR"], "Metric"]:
            md = d[d["_Metric"] == metric].copy()
            temp = []
            for i in range(len(condition_order) - 1):
                for j in range(i + 1, len(condition_order)):
                    a, b = condition_order[i], condition_order[j]
                    pt = _pairwise_lrt(md, a, b)
                    temp.append({"Metric": metric, "Comparison": f"{b} − {a}",
                                 "beta": pt["beta"], "SE": pt["SE"],
                                 "95% CI Low": pt["CI Low"], "95% CI High": pt["CI High"],
                                 "LR chi-square": pt["chi2"], "df": pt["df"], "Raw p": pt["p"]})
            temp = pd.DataFrame(temp)
            if not temp.empty:
                valid_ph = temp["Raw p"].notna()
                temp["Holm-adjusted p"] = np.nan
                temp["Significant After Holm"] = False
                if valid_ph.any():
                    rej, hp, _, _ = multipletests(temp.loc[valid_ph, "Raw p"], alpha=ALPHA, method="holm")
                    temp.loc[valid_ph, "Holm-adjusted p"] = hp
                    temp.loc[valid_ph, "Significant After Holm"] = rej
                posthoc_rows.extend(temp.to_dict("records"))

    posthoc = pd.DataFrame(posthoc_rows)

    def sig_text(row):
        return f"χ²({int(row['df'])}) = {row['LR chi-square']:.3f}; p {_fmt_p(row['Raw p'])}; q {_fmt_p(row['FDR q'])}"
    results["Significance"] = results.apply(sig_text, axis=1)

    paper_cols = ["Metric", "Participants"] + list(condition_order)
    if len(condition_order) == 2:
        paper_cols += ["beta", "SE", "95% CI Low", "95% CI High"]
    paper_cols += ["Significance"]
    paper_table = results[paper_cols].copy()

    metadata = pd.DataFrame({
        "Item": ["Primary model", "Experimental unit", "Repeated-trial handling", "Omnibus inference",
                 "Multiple-comparison correction", "Assumption checks", "Fallback/sensitivity rule"],
        "Specification": [
            "Linear mixed-effects model: Outcome ~ Condition + (1 | Participant)",
            "Participant",
            f"All valid trial-level repetitions are retained in the LMM; participant-condition means require exactly {expected_repetitions} valid trials and are used only for descriptive statistics and sensitivity analyses.",
            "ML full-vs-reduced likelihood-ratio test; df equals the difference in fixed-effect parameters.",
            "Benjamini-Hochberg FDR across the prespecified outcomes in this table; Holm adjustment across post-hoc pairwise LMM likelihood-ratio tests when the omnibus test survives FDR.",
            "Shapiro-Wilk test of LMM residuals and Brown-Forsythe test of residual variance across conditions.",
            "If residual normality/homogeneity is violated, the LMM is singular, or convergence fails, a participant-level Wilcoxon signed-rank test (2 conditions) or Friedman test (>2 conditions) is reported as a sensitivity analysis; the LMM remains the prespecified primary model.",
        ],
    })

    output_path = OUTPUT_DIR / output_filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        paper_table.to_excel(writer, sheet_name="Paper Table", index=False)
        results.to_excel(writer, sheet_name="Full LMM Output", index=False)
        diagnostics.to_excel(writer, sheet_name="Model Diagnostics", index=False)
        sensitivity.to_excel(writer, sheet_name="Sensitivity Analysis", index=False)
        if not posthoc.empty:
            posthoc.to_excel(writer, sheet_name="Post Hoc Tests", index=False)
        pc.to_excel(writer, sheet_name="Participant Means", index=False)
        d.to_excel(writer, sheet_name="LMM Input Data", index=False)
        metadata.to_excel(writer, sheet_name="Analysis Metadata", index=False)
    _format_workbook(output_path)
    return paper_table, results, diagnostics, sensitivity, posthoc


def _format_workbook(path):
    wb = load_workbook(path)
    thin = Side(style="thin", color="B7B7B7")
    fill = PatternFill(fill_type="solid", fgColor="D9EAF7")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="center", wrap_text=True)
                cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for col in ws.columns:
            letter = col[0].column_letter
            max_len = min(60, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col) + 2))
            ws.column_dimensions[letter].width = max_len
    wb.save(path)


## Table: Sample Entropy

In [ ]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency, Hz

participants = [
    f"P{i:03d}"
    for i in range(1, 17)
]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}

# Sample entropy parameters
EMBEDDING_DIMENSION = 2
TOLERANCE_RATIO = 0.20
MIN_SIGNAL_LENGTH = 50


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# SAMPLE ENTROPY
# =====================================================

def sample_entropy(
    signal,
    m=EMBEDDING_DIMENSION,
    r_ratio=TOLERANCE_RATIO,
    min_length=MIN_SIGNAL_LENGTH,
):
    """
    Calculate sample entropy using Chebyshev distance.

    The same eligible starting positions are used for the
    length-m and length-(m+1) template comparisons.

    Parameters
    ----------
    signal : array-like
        One-dimensional signal.

    m : int
        Embedding dimension.

    r_ratio : float
        Tolerance expressed as a proportion of the signal
        standard deviation.

    min_length : int
        Minimum acceptable number of finite observations.

    Returns
    -------
    float
        Sample entropy value.

        Larger values indicate greater irregularity or
        lower predictability.

        Returns np.nan when the calculation cannot be
        completed.
    """

    signal = np.asarray(
        signal,
        dtype=float,
    )

    # Remove nonfinite values
    signal = signal[
        np.isfinite(signal)
    ]

    n = len(signal)

    if n < min_length:
        return np.nan

    if m < 1:
        raise ValueError(
            "Embedding dimension must be at least 1."
        )

    if r_ratio <= 0:
        raise ValueError(
            "Tolerance ratio must be greater than 0."
        )

    signal_sd = np.std(
        signal,
        ddof=1,
    )

    if (
        not np.isfinite(signal_sd)
        or signal_sd <= 0
    ):
        return np.nan

    tolerance = (
        r_ratio * signal_sd
    )

    # Use the same number of starting positions for
    # m-length and m+1-length templates.
    #
    # Starting indices:
    # 0 through n - m - 1
    #
    # Number of templates:
    # n - m
    number_of_templates = n - m

    if number_of_templates < 2:
        return np.nan

    templates_m = np.array(
        [
            signal[i:i + m]
            for i in range(
                number_of_templates
            )
        ],
        dtype=float,
    )

    templates_m1 = np.array(
        [
            signal[i:i + m + 1]
            for i in range(
                number_of_templates
            )
        ],
        dtype=float,
    )

    matches_m = 0
    matches_m1 = 0

    # Compare only unique template pairs:
    # j > i.
    #
    # This excludes self-matches and avoids counting
    # every pair twice.
    for i in range(
        number_of_templates - 1
    ):

        comparison_m = (
            templates_m[i + 1:]
            - templates_m[i]
        )

        distances_m = np.max(
            np.abs(comparison_m),
            axis=1,
        )

        comparison_m1 = (
            templates_m1[i + 1:]
            - templates_m1[i]
        )

        distances_m1 = np.max(
            np.abs(comparison_m1),
            axis=1,
        )

        matches_m += int(
            np.sum(
                distances_m <= tolerance
            )
        )

        matches_m1 += int(
            np.sum(
                distances_m1 <= tolerance
            )
        )

    if matches_m == 0:
        return np.nan

    if matches_m1 == 0:
        return np.nan

    sampen = -np.log(
        matches_m1 / matches_m
    )

    return float(sampen)


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    """
    Format p-values for manuscript reporting.
    """

    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return (
        f"= {p:.3f}"
        .replace(
            "0.",
            ".",
        )
    )


def mean_sd_ci(values):
    """
    Return mean ± SD and the 95% confidence interval
    of the condition mean.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(
        ddof=1
    )

    standard_error = (
        sd_value / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_value
        - t_critical * standard_error
    )

    ci_high = (
        mean_value
        + t_critical * standard_error
    )

    return (
        f"{mean_value:.3f} ± "
        f"{sd_value:.3f}\n"
        f"[{ci_low:.3f}, "
        f"{ci_high:.3f}]"
    )


def paired_difference_ci(
    differences,
):
    """
    Return the mean paired difference and its
    95% confidence interval.

    Difference direction:
    HIGH minus LOW.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    n = len(differences)

    if n < 2:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    mean_difference = (
        differences.mean()
    )

    sd_difference = (
        differences.std(
            ddof=1
        )
    )

    standard_error = (
        sd_difference
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_difference
        - t_critical * standard_error
    )

    ci_high = (
        mean_difference
        + t_critical * standard_error
    )

    return (
        mean_difference,
        ci_low,
        ci_high,
    )


def cohen_dz(
    differences,
):
    """
    Cohen's dz for paired HIGH-minus-LOW
    differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = (
        differences.std(
            ddof=1
        )
    )

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(
    differences,
):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values
    under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    # Wilcoxon excludes zero differences
    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(
            differences
        )
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# CALCULATE TRIAL-LEVEL SAMPLE ENTROPY
# =====================================================

trial_rows = []
missing_files = []
processing_errors = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]


for participant in participants:

    for condition, trials in (
        conditions.items()
    ):

        for trial in trials:

            file_name = (
                f"{participant}-"
                f"{trial:03d}.xlsx"
            )

            if not os.path.exists(
                file_name
            ):
                missing_files.append(
                    file_name
                )
                continue

            try:
                trial_data = (
                    pd.read_excel(
                        file_name,
                        sheet_name=(
                            "Center of Mass"
                        ),
                    )
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column
                    in required_columns
                    if column
                    not in trial_data.columns
                ]

                if missing_columns:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Missing columns: "
                                + ", ".join(
                                    missing_columns
                                )
                            ),
                        }
                    )
                    continue

                timing_index = (
                    trial - 3
                )

                if participant in cut_times:

                    start_time_seconds = (
                        cut_times[
                            participant
                        ][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = len(
                        trial_data
                    )

                    end_time_seconds = (
                        end_frame / fs
                    )

                else:

                    start_time_seconds = (
                        reaction_times[
                            participant
                        ]["start"][
                            timing_index
                        ]
                    )

                    end_time_seconds = (
                        reaction_times[
                            participant
                        ]["end"][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds
                            * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if (
                    end_frame
                    <= start_frame
                ):
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Invalid segment: "
                                f"start={start_frame}, "
                                f"end={end_frame}"
                            ),
                        }
                    )
                    continue

                segment = (
                    trial_data.iloc[
                        start_frame:
                        end_frame
                    ]
                    .copy()
                )

                segment_samples = len(
                    segment
                )

                segment_duration = (
                    segment_samples
                    / fs
                )

                position_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM pos x"
                        ] ** 2
                        + segment[
                            "CoM pos y"
                        ] ** 2
                        + segment[
                            "CoM pos z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                velocity_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM vel x"
                        ] ** 2
                        + segment[
                            "CoM vel y"
                        ] ** 2
                        + segment[
                            "CoM vel z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                acceleration_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM acc x"
                        ] ** 2
                        + segment[
                            "CoM acc y"
                        ] ** 2
                        + segment[
                            "CoM acc z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                position_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            position_magnitude
                        )
                    )
                )

                velocity_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            velocity_magnitude
                        )
                    )
                )

                acceleration_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            acceleration_magnitude
                        )
                    )
                )

                position_sd = np.nanstd(
                    position_magnitude,
                    ddof=1,
                )

                velocity_sd = np.nanstd(
                    velocity_magnitude,
                    ddof=1,
                )

                acceleration_sd = np.nanstd(
                    acceleration_magnitude,
                    ddof=1,
                )

                trial_rows.append(
                    {
                        "Participant":
                            participant,
                        "Condition":
                            condition,
                        "Trial":
                            trial,
                        "File":
                            file_name,
                        "Sampling Frequency Hz":
                            fs,
                        "Start Time s":
                            start_time_seconds,
                        "End Time s":
                            end_time_seconds,
                        "Start Frame":
                            start_frame,
                        "End Frame":
                            end_frame,
                        "Segment Samples":
                            segment_samples,
                        "Segment Duration s":
                            segment_duration,
                        "Embedding Dimension":
                            EMBEDDING_DIMENSION,
                        "Tolerance Ratio":
                            TOLERANCE_RATIO,
                        "Minimum Signal Length":
                            MIN_SIGNAL_LENGTH,
                        "Position Valid Samples":
                            position_valid_samples,
                        "Velocity Valid Samples":
                            velocity_valid_samples,
                        "Acceleration Valid Samples":
                            acceleration_valid_samples,
                        "Position SD":
                            position_sd,
                        "Velocity SD":
                            velocity_sd,
                        "Acceleration SD":
                            acceleration_sd,
                        "Position Tolerance":
                            (
                                TOLERANCE_RATIO
                                * position_sd
                            ),
                        "Velocity Tolerance":
                            (
                                TOLERANCE_RATIO
                                * velocity_sd
                            ),
                        "Acceleration Tolerance":
                            (
                                TOLERANCE_RATIO
                                * acceleration_sd
                            ),
                        "SampEn_Position":
                            sample_entropy(
                                position_magnitude
                            ),
                        "SampEn_Velocity":
                            sample_entropy(
                                velocity_magnitude
                            ),
                        "SampEn_Acceleration":
                            sample_entropy(
                                acceleration_magnitude
                            ),
                    }
                )

            except Exception as error:

                processing_errors.append(
                    {
                        "File": file_name,
                        "Error": str(error),
                    }
                )


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No sample entropy values were "
        "calculated. Check file paths, "
        "file names, worksheets, and "
        "required columns."
    )


sample_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["SampEn_Position", "SampEn_Velocity", "SampEn_Acceleration"],
    var_name="Metric", value_name="Value",
)
sample_long["Metric"] = sample_long["Metric"].map({
    "SampEn_Position": "Position", "SampEn_Velocity": "Velocity", "SampEn_Acceleration": "Acceleration"
})
run_lmm_family(sample_long, "Metric", "Value", "Participant", "Condition", ["LOW", "HIGH"],
               "Table_Sample_Entropy.xlsx", decimals=3)


## Table: Lyapunov Exponent

In [ ]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency in Hz

# Rosenstein largest Lyapunov exponent parameters
embedding_dimension = 5
time_delay_samples = 10          # delay-embedding lag
theiler_window_samples = 60      # exclude neighbors within ±1.0 s
trajectory_length_samples = 20   # divergence trajectory length
fit_offset_samples = 0           # first divergence point included in fit
fit_method = "poly"              # ordinary least-squares line fit

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# LARGEST LYAPUNOV EXPONENT: ROSENSTEIN METHOD
# =====================================================

def largest_lyapunov(
    signal,
    m=embedding_dimension,
    lag=time_delay_samples,
    fs=fs,
    min_tsep=theiler_window_samples,
    trajectory_len=trajectory_length_samples,
    fit_offset=fit_offset_samples,
    fit=fit_method,
):
    """
    Estimate the largest Lyapunov exponent with the Rosenstein method
    using the validated nolds.lyap_r implementation.

    Parameters
    ----------
    signal : array-like
        One-dimensional time series.
    m : int
        Embedding dimension.
    lag : int
        Delay between coordinates of each embedded vector, in samples.
    fs : float
        Sampling frequency in Hz.
    min_tsep : int
        Theiler window/minimum temporal separation between neighboring
        state-space vectors, in samples.
    trajectory_len : int
        Number of forward samples used to construct the mean logarithmic
        divergence trajectory.
    fit_offset : int
        Number of initial divergence samples excluded from the linear fit.
    fit : {"poly", "RANSAC"}
        Line-fitting method. "poly" gives an ordinary least-squares fit.

    Returns
    -------
    float
        Largest Lyapunov exponent in s^-1. Returns np.nan when the signal
        is too short or the estimate cannot be computed.
    """
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]

    if len(signal) == 0 or fs <= 0:
        return np.nan

    if np.nanstd(signal, ddof=1) == 0:
        return np.nan

    # nolds reports the minimum length implied by the selected embedding,
    # Theiler window, and divergence-trajectory settings.
    minimum_required = nolds.lyap_r_len(
        emb_dim=m,
        lag=lag,
        min_tsep=min_tsep,
        trajectory_len=trajectory_len,
    )

    if len(signal) < minimum_required:
        return np.nan

    try:
        exponent = nolds.lyap_r(
            signal,
            emb_dim=m,
            lag=lag,
            min_tsep=min_tsep,
            tau=1.0 / fs,          # converts slope from per-sample to s^-1
            trajectory_len=trajectory_len,
            fit=fit,
            fit_offset=fit_offset,
            debug_plot=False,
            debug_data=False,
        )
    except (ValueError, RuntimeError, FloatingPointError):
        return np.nan

    return float(exponent) if np.isfinite(exponent) else np.nan


def lyapunov_debug_data(
    signal,
    m=embedding_dimension,
    lag=time_delay_samples,
    fs=fs,
    min_tsep=theiler_window_samples,
    trajectory_len=trajectory_length_samples,
    fit_offset=fit_offset_samples,
    fit=fit_method,
):
    """
    Return the LLE and divergence-curve data for diagnostic plotting.

    Returns
    -------
    exponent : float
    time_seconds : ndarray
    mean_log_divergence : ndarray
    fitted_line : ndarray
    """
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]

    minimum_required = nolds.lyap_r_len(
        emb_dim=m,
        lag=lag,
        min_tsep=min_tsep,
        trajectory_len=trajectory_len,
    )

    if len(signal) < minimum_required:
        return np.nan, np.array([]), np.array([]), np.array([])

    try:
        exponent, debug = nolds.lyap_r(
            signal,
            emb_dim=m,
            lag=lag,
            min_tsep=min_tsep,
            tau=1.0 / fs,
            trajectory_len=trajectory_len,
            fit=fit,
            fit_offset=fit_offset,
            debug_plot=False,
            debug_data=True,
        )
    except (ValueError, RuntimeError, FloatingPointError):
        return np.nan, np.array([]), np.array([]), np.array([])

    ks, divergence, coefficients = debug
    time_seconds = np.asarray(ks, dtype=float) / fs
    divergence = np.asarray(divergence, dtype=float)
    fitted_line = np.polyval(coefficients, np.asarray(ks, dtype=float))

    return float(exponent), time_seconds, divergence, fitted_line


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% confidence interval.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL LLE
# =====================================================

trial_rows = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column in required_columns
                    if column not in trial_data.columns
                ]

                if missing_columns:
                    continue

                timing_index = trial - 3

                if participant in cut_times:
                    start_time_seconds = (
                        cut_times[participant][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds * fs
                        )
                    )

                    end_frame = len(trial_data)
                    end_time_seconds = end_frame / fs

                else:
                    start_time_seconds = (
                        reaction_times[participant][
                            "start"
                        ][timing_index]
                    )

                    end_time_seconds = (
                        reaction_times[participant][
                            "end"
                        ][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if end_frame <= start_frame:
                    continue

                segment = trial_data.iloc[
                    start_frame:end_frame
                ].copy()

                # -------------------------------------
                # DATA-LENGTH INFORMATION
                # -------------------------------------

                segment_length_samples = len(segment)

                segment_duration_seconds = (
                    segment_length_samples / fs
                )

                embedded_length = (
                    segment_length_samples
                    - (embedding_dimension - 1) * time_delay_samples
                )

                minimum_required_samples = nolds.lyap_r_len(
                    emb_dim=embedding_dimension,
                    lag=time_delay_samples,
                    min_tsep=theiler_window_samples,
                    trajectory_len=trajectory_length_samples,
                )

                if embedded_length <= 0:
                    continue

                position_magnitude = np.sqrt(
                    segment["CoM pos x"] ** 2
                    + segment["CoM pos y"] ** 2
                    + segment["CoM pos z"] ** 2
                )

                velocity_magnitude = np.sqrt(
                    segment["CoM vel x"] ** 2
                    + segment["CoM vel y"] ** 2
                    + segment["CoM vel z"] ** 2
                )

                acceleration_magnitude = np.sqrt(
                    segment["CoM acc x"] ** 2
                    + segment["CoM acc y"] ** 2
                    + segment["CoM acc z"] ** 2
                )

                position_valid_samples = (
                    np.isfinite(
                        position_magnitude
                    ).sum()
                )

                velocity_valid_samples = (
                    np.isfinite(
                        velocity_magnitude
                    ).sum()
                )

                acceleration_valid_samples = (
                    np.isfinite(
                        acceleration_magnitude
                    ).sum()
                )

                trial_rows.append(
                    {
                        "Participant": participant,
                        "Condition": condition,
                        "Trial": trial,
                        "File": file_name,
                        "Sampling_Frequency_Hz": fs,
                        "Start_Time_Seconds":
                            start_time_seconds,
                        "End_Time_Seconds":
                            end_time_seconds,
                        "Start_Frame": start_frame,
                        "End_Frame": end_frame,
                        "Segment_Length_Samples":
                            segment_length_samples,
                        "Segment_Duration_Seconds":
                            segment_duration_seconds,
                        "Embedding_Dimension":
                            embedding_dimension,
                        "Time_Delay_Samples":
                            time_delay_samples,
                        "Time_Delay_Seconds":
                            time_delay_samples / fs,
                        "Embedded_Length":
                            embedded_length,
                        "Theiler_Window_Samples":
                            theiler_window_samples,
                        "Theiler_Window_Seconds":
                            theiler_window_samples / fs,
                        "Trajectory_Length_Samples":
                            trajectory_length_samples,
                        "Trajectory_Length_Seconds":
                            trajectory_length_samples / fs,
                        "Fit_Offset_Samples":
                            fit_offset_samples,
                        "Fit_Method":
                            fit_method,
                        "Minimum_Required_Samples":
                            minimum_required_samples,
                        "Position_Valid_Samples":
                            position_valid_samples,
                        "Velocity_Valid_Samples":
                            velocity_valid_samples,
                        "Acceleration_Valid_Samples":
                            acceleration_valid_samples,
                        "Lyap_CoM_Pos":
                            largest_lyapunov(
                                position_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                        "Lyap_CoM_Vel":
                            largest_lyapunov(
                                velocity_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                        "Lyap_CoM_Acc":
                            largest_lyapunov(
                                acceleration_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                    }
                )

            except Exception as error:
                continue


trial_level = pd.DataFrame(trial_rows)

if trial_level.empty:
    raise ValueError(
        "No LLE data were computed. "
        "Check the working directory and "
        "input workbook structure."
    )


lle_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["Lyap_CoM_Pos", "Lyap_CoM_Vel", "Lyap_CoM_Acc"],
    var_name="Metric", value_name="Value",
)
lle_long["Metric"] = lle_long["Metric"].map({
    "Lyap_CoM_Pos": "Position", "Lyap_CoM_Vel": "Velocity", "Lyap_CoM_Acc": "Acceleration"
})
run_lmm_family(lle_long, "Metric", "Value", "Participant", "Condition", ["LOW", "HIGH"],
               "Table_Lyapunov_Exponent.xlsx", decimals=3)


## Table: Smoothness (SPARC)

In [ ]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency, Hz

participants = [
    f"P{i:03d}"
    for i in range(1, 17)
]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}

# Published SPARC reference settings
SPARC_PADLEVEL = 4
SPARC_MAX_CUTOFF_HZ = 10.0
SPARC_AMPLITUDE_THRESHOLD = 0.05
MIN_SIGNAL_LENGTH = 60


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# SPARC SMOOTHNESS
# =====================================================

def sparc(
    movement,
    fs,
    padlevel=SPARC_PADLEVEL,
    max_cutoff_hz=SPARC_MAX_CUTOFF_HZ,
    amplitude_threshold=SPARC_AMPLITUDE_THRESHOLD,
    min_length=MIN_SIGNAL_LENGTH,
):
    """
    Calculate SPARC movement smoothness using the published
    reference implementation.

    Parameters
    ----------
    movement : array-like
        One-dimensional movement profile. A speed profile is
        most consistent with the original SPARC formulation.

    fs : float
        Sampling frequency in Hz.

    padlevel : int
        Zero-padding level. The FFT length equals the next
        power of two multiplied by 2**padlevel.

    max_cutoff_hz : float
        Maximum frequency considered in the calculation.

    amplitude_threshold : float
        Normalized magnitude-spectrum threshold used to select
        the adaptive cutoff frequency.

    min_length : int
        Minimum number of finite observations.

    Returns
    -------
    sparc_value : float
        SPARC smoothness value. Values closer to zero indicate
        smoother movement. More negative values indicate less
        smooth movement.

    cutoff_frequency : float
        Adaptive cutoff frequency used for the arc-length
        calculation.

    selected_bins : int
        Number of spectral bins used in the calculation.

    nfft : int
        FFT length after zero padding.
    """

    movement = np.asarray(
        movement,
        dtype=float,
    )

    movement = movement[
        np.isfinite(movement)
    ]

    if len(movement) < min_length:
        return np.nan, np.nan, 0, np.nan

    if fs <= 0:
        raise ValueError(
            "Sampling frequency must be greater than zero."
        )

    if padlevel < 0:
        raise ValueError(
            "SPARC pad level cannot be negative."
        )

    if (
        max_cutoff_hz <= 0
        or max_cutoff_hz > fs / 2
    ):
        raise ValueError(
            "The maximum cutoff frequency must be greater "
            "than zero and no greater than the Nyquist frequency."
        )

    if not 0 < amplitude_threshold < 1:
        raise ValueError(
            "The SPARC amplitude threshold must be between "
            "zero and one."
        )

    # Do not mean-center the signal. The published reference
    # implementation calculates SPARC directly from the
    # movement profile.

    # FFT length with zero padding
    nfft = int(
        2 ** (
            np.ceil(
                np.log2(
                    len(movement)
                )
            )
            + padlevel
        )
    )

    # Frequency vector used by the reference implementation
    frequencies = np.arange(
        0,
        fs,
        fs / nfft,
    )

    # Magnitude spectrum
    magnitude_spectrum = np.abs(
        np.fft.fft(
            movement,
            n=nfft,
        )
    )

    maximum_magnitude = np.max(
        magnitude_spectrum
    )

    if (
        not np.isfinite(maximum_magnitude)
        or maximum_magnitude <= 0
    ):
        return np.nan, np.nan, 0, nfft

    # Normalize magnitude spectrum
    normalized_spectrum = (
        magnitude_spectrum
        / maximum_magnitude
    )

    # First restrict the spectrum to the maximum cutoff
    maximum_cutoff_indices = np.where(
        frequencies <= max_cutoff_hz
    )[0]

    if len(maximum_cutoff_indices) < 2:
        return np.nan, np.nan, 0, nfft

    selected_frequencies = frequencies[
        maximum_cutoff_indices
    ]

    selected_spectrum = normalized_spectrum[
        maximum_cutoff_indices
    ]

    # Identify all bins at or above the amplitude threshold
    above_threshold_indices = np.where(
        selected_spectrum
        >= amplitude_threshold
    )[0]

    if len(above_threshold_indices) < 2:
        return np.nan, np.nan, 0, nfft

    # Published implementation retains the interval from the
    # first through the final above-threshold spectral bin.
    first_index = above_threshold_indices[0]
    last_index = above_threshold_indices[-1]

    selected_frequencies = selected_frequencies[
        first_index:last_index + 1
    ]

    selected_spectrum = selected_spectrum[
        first_index:last_index + 1
    ]

    if len(selected_frequencies) < 2:
        return np.nan, np.nan, 0, nfft

    frequency_range = (
        selected_frequencies[-1]
        - selected_frequencies[0]
    )

    if frequency_range <= 0:
        return np.nan, np.nan, 0, nfft

    # Normalize the frequency axis and calculate the negative
    # spectral arc length.
    normalized_frequency_difference = (
        np.diff(selected_frequencies)
        / frequency_range
    )

    spectrum_difference = np.diff(
        selected_spectrum
    )

    arc_length_elements = np.sqrt(
        normalized_frequency_difference ** 2
        + spectrum_difference ** 2
    )

    sparc_value = -np.sum(
        arc_length_elements
    )

    cutoff_frequency = (
        selected_frequencies[-1]
    )

    selected_bins = len(
        selected_frequencies
    )

    return (
        float(sparc_value),
        float(cutoff_frequency),
        int(selected_bins),
        int(nfft),
    )


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace(
        "0.",
        ".",
    )


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(
        ddof=1
    )

    standard_error = (
        sd_value
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_value
        - t_critical * standard_error
    )

    ci_high = (
        mean_value
        + t_critical * standard_error
    )

    return (
        f"{mean_value:.3f} ± "
        f"{sd_value:.3f}\n"
        f"[{ci_low:.3f}, "
        f"{ci_high:.3f}]"
    )


def paired_difference_ci(differences):
    """
    Return mean HIGH-minus-LOW difference and 95% CI.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    n = len(differences)

    if n < 2:
        return np.nan, np.nan, np.nan

    mean_difference = differences.mean()

    sd_difference = differences.std(
        ddof=1
    )

    standard_error = (
        sd_difference
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_difference
        - t_critical * standard_error
    )

    ci_high = (
        mean_difference
        + t_critical * standard_error
    )

    return (
        mean_difference,
        ci_low,
        ci_high,
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(
        ddof=1
    )

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(
            differences
        )
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL SPARC
# =====================================================

trial_rows = []
missing_files = []
processing_errors = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]


for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-"
                f"{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                missing_files.append(
                    file_name
                )
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column in required_columns
                    if column not in trial_data.columns
                ]

                if missing_columns:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Missing columns: "
                                + ", ".join(
                                    missing_columns
                                )
                            ),
                        }
                    )
                    continue

                timing_index = trial - 3

                if participant in cut_times:

                    start_time_seconds = (
                        cut_times[
                            participant
                        ][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = len(
                        trial_data
                    )

                    end_time_seconds = (
                        end_frame / fs
                    )

                else:

                    start_time_seconds = (
                        reaction_times[
                            participant
                        ]["start"][timing_index]
                    )

                    end_time_seconds = (
                        reaction_times[
                            participant
                        ]["end"][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds
                            * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if end_frame <= start_frame:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Invalid segment: "
                                f"start={start_frame}, "
                                f"end={end_frame}"
                            ),
                        }
                    )
                    continue

                segment = trial_data.iloc[
                    start_frame:end_frame
                ].copy()

                segment_samples = len(
                    segment
                )

                segment_duration_seconds = (
                    segment_samples / fs
                )

                position_magnitude = np.sqrt(
                    segment["CoM pos x"] ** 2
                    + segment["CoM pos y"] ** 2
                    + segment["CoM pos z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                velocity_magnitude = np.sqrt(
                    segment["CoM vel x"] ** 2
                    + segment["CoM vel y"] ** 2
                    + segment["CoM vel z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                acceleration_magnitude = np.sqrt(
                    segment["CoM acc x"] ** 2
                    + segment["CoM acc y"] ** 2
                    + segment["CoM acc z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                (
                    position_sparc,
                    position_cutoff,
                    position_bins,
                    position_nfft,
                ) = sparc(
                    position_magnitude,
                    fs=fs,
                )

                (
                    velocity_sparc,
                    velocity_cutoff,
                    velocity_bins,
                    velocity_nfft,
                ) = sparc(
                    velocity_magnitude,
                    fs=fs,
                )

                (
                    acceleration_sparc,
                    acceleration_cutoff,
                    acceleration_bins,
                    acceleration_nfft,
                ) = sparc(
                    acceleration_magnitude,
                    fs=fs,
                )

                trial_rows.append(
                    {
                        "Participant":
                            participant,
                        "Condition":
                            condition,
                        "Trial":
                            trial,
                        "File":
                            file_name,
                        "Sampling Frequency Hz":
                            fs,
                        "Start Time s":
                            start_time_seconds,
                        "End Time s":
                            end_time_seconds,
                        "Start Frame":
                            start_frame,
                        "End Frame":
                            end_frame,
                        "Segment Samples":
                            segment_samples,
                        "Segment Duration s":
                            segment_duration_seconds,
                        "SPARC Pad Level":
                            SPARC_PADLEVEL,
                        "Maximum Cutoff Hz":
                            SPARC_MAX_CUTOFF_HZ,
                        "Amplitude Threshold":
                            SPARC_AMPLITUDE_THRESHOLD,
                        "SPARC_Position":
                            position_sparc,
                        "Position Cutoff Hz":
                            position_cutoff,
                        "Position Selected Bins":
                            position_bins,
                        "Position NFFT":
                            position_nfft,
                        "SPARC_Velocity":
                            velocity_sparc,
                        "Velocity Cutoff Hz":
                            velocity_cutoff,
                        "Velocity Selected Bins":
                            velocity_bins,
                        "Velocity NFFT":
                            velocity_nfft,
                        "SPARC_Acceleration":
                            acceleration_sparc,
                        "Acceleration Cutoff Hz":
                            acceleration_cutoff,
                        "Acceleration Selected Bins":
                            acceleration_bins,
                        "Acceleration NFFT":
                            acceleration_nfft,
                    }
                )

            except Exception as error:
                processing_errors.append(
                    {
                        "File": file_name,
                        "Error": str(error),
                    }
                )


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No SPARC values were calculated. "
        "Check file paths, worksheets, and column names."
    )


sparc_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["SPARC_Position", "SPARC_Velocity", "SPARC_Acceleration"],
    var_name="Metric", value_name="Value",
)
sparc_long["Metric"] = sparc_long["Metric"].map({
    "SPARC_Position": "Position", "SPARC_Velocity": "Velocity", "SPARC_Acceleration": "Acceleration"
})
run_lmm_family(sparc_long, "Metric", "Value", "Participant", "Condition", ["LOW", "HIGH"],
               "Table_Smoothness_SPARC.xlsx", decimals=3)


## Table: Pelvis–Foot Redistribution

In [ ]:
# =====================================================
# CONFIGURATION
# =====================================================
fs = 60  # Hz

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================
cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# ANALYSIS SETTINGS
# =====================================================
# The transient active-guidance phase analyzed in the paper.
PHASES_TO_RUN = ["P2"]

# Two prespecified pelvis-to-foot redistribution outcomes.
DISTAL_GROUPS_TO_RUN = [
    "Right_Foot",
    "Left_Foot",
]

VARIABLE_TO_RUN = "Redistribution_Index"


# =====================================================
# SEGMENT GROUPS
# =====================================================
distal_groups = {
    "Bilateral_UpperLeg": [
        "Right Upper Leg",
        "Left Upper Leg",
    ],
    "Bilateral_LowerLeg": [
        "Right Lower Leg",
        "Left Lower Leg",
    ],
    "Bilateral_Foot": [
        "Right Foot",
        "Left Foot",
    ],
    "Right_UpperLeg": ["Right Upper Leg"],
    "Left_UpperLeg": ["Left Upper Leg"],
    "Right_LowerLeg": ["Right Lower Leg"],
    "Left_LowerLeg": ["Left Lower Leg"],
    "Right_Foot": ["Right Foot"],
    "Left_Foot": ["Left Foot"],
}


# =====================================================
# CALCULATION FUNCTIONS
# =====================================================
def acceleration_magnitude(data, segment_name):
    return np.sqrt(
        data[f"{segment_name} x"] ** 2
        + data[f"{segment_name} y"] ** 2
        + data[f"{segment_name} z"] ** 2
    )


def mean_acceleration(data, segment_name):
    values = acceleration_magnitude(
        data,
        segment_name,
    )

    return values.replace(
        [np.inf, -np.inf],
        np.nan,
    ).mean()


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return differences.mean() / sd_difference


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# CALCULATE TRIAL-LEVEL REDISTRIBUTION INDICES
# =====================================================
trial_rows = []

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Segment Acceleration",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                timing_index = trial - 3

                if participant in cut_times:
                    start_p2 = int(round(
                        cut_times[participant][timing_index]
                        * fs
                    ))
                    end_p2 = len(trial_data)
                    start_p3 = None
                else:
                    start_p2 = int(round(
                        reaction_times[participant]["start"][
                            timing_index
                        ] * fs
                    ))
                    end_p2 = int(round(
                        reaction_times[participant]["end"][
                            timing_index
                        ] * fs
                    ))
                    start_p3 = end_p2

                start_p2 = max(
                    0,
                    start_p2,
                )
                end_p2 = min(
                    len(trial_data),
                    end_p2,
                )

                if end_p2 <= start_p2:
                    continue

                phase_segments = {
                    "P2": trial_data.iloc[
                        start_p2:end_p2
                    ],
                }

                if start_p3 is not None:
                    phase_segments["P3"] = (
                        trial_data.iloc[start_p3:]
                    )

                for phase, segment_data in (
                    phase_segments.items()
                ):

                    if segment_data.empty:
                        continue

                    pelvis_mean = mean_acceleration(
                        segment_data,
                        "Pelvis",
                    )

                    for (
                        distal_group,
                        segment_names,
                    ) in distal_groups.items():

                        distal_values = [
                            mean_acceleration(
                                segment_data,
                                segment_name,
                            )
                            for segment_name
                            in segment_names
                        ]

                        distal_mean = np.nanmean(
                            distal_values
                        )

                        if (
                            not np.isfinite(pelvis_mean)
                            or not np.isfinite(distal_mean)
                            or distal_mean == 0
                        ):
                            redistribution_index = np.nan
                        else:
                            redistribution_index = (
                                pelvis_mean
                                / distal_mean
                            )

                        trial_rows.append({
                            "Participant": participant,
                            "Condition": condition,
                            "Trial": trial,
                            "Phase": phase,
                            "Distal_Group": distal_group,
                            "Pelvis_Mean_Acc": pelvis_mean,
                            "Distal_Mean_Acc": distal_mean,
                            "Redistribution_Index":
                                redistribution_index,
                        })

            except Exception:
                continue


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No redistribution-index data were computed. "
        "Check the working directory and workbook structure."
    )


pf_long = trial_level[(trial_level["Phase"].isin(PHASES_TO_RUN)) & (trial_level["Distal_Group"].isin(DISTAL_GROUPS_TO_RUN))].copy()
pf_long["Metric"] = pf_long["Distal_Group"].map({"Right_Foot": "Redistribution Index (pelvis/right foot)",
                                                   "Left_Foot": "Redistribution Index (pelvis/left foot)"})
run_lmm_family(pf_long, "Metric", VARIABLE_TO_RUN, "Participant", "Condition", ["LOW", "HIGH"],
               "Table_Pelvis_Foot_Redistribution.xlsx", decimals=3)


## SPM: Pelvis–Foot Redistribution Trajectories

In [ ]:
# Input parameters and timing windows
fs = 60
participants = [f"P{i:03d}" for i in range(1, 17)]
trial_map = {3: "LOW", 4: "LOW", 5: "HIGH", 6: "HIGH"}
target_trials = [3, 4, 5, 6]
sheet_name = "Segment Acceleration"
n_points = 101
spm_outcomes = ["Pelvis/Left Foot Redistribution", "Pelvis/Right Foot Redistribution"]
spm_alpha_family = 0.05
spm_alpha_per_outcome = spm_alpha_family / len(spm_outcomes)  # Bonferroni across the two prespecified SPM outcomes

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}
reaction_times = {
    "P003": {"start": [10.28, 13.53, 7.50, 8.15], "end": [21.50, 23.80, 13.50, 12.50]},
    "P004": {"start": [8.31, 12.50, 5.88, 5.18], "end": [18.00, 22.00, 11.40, 9.80]},
    "P005": {"start": [12.16, 11.78, 5.58, 6.03], "end": [20.60, 20.80, 10.00, 10.50]},
    "P006": {"start": [10.66, 12.90, 5.81, 6.25], "end": [19.70, 20.50, 7.20, 9.80]},
    "P007": {"start": [10.00, 12.00, 6.10, 6.00], "end": [23.70, 15.60, 8.00, 9.60]},
    "P008": {"start": [11.00, 12.00, 6.00, 7.00], "end": [19.60, 24.00, 11.80, 10.20]},
    "P009": {"start": [10.29, 10.56, 5.86, 6.02], "end": [20.50, 23.00, 9.00, 12.00]},
    "P010": {"start": [12.00, 12.00, 6.00, 16.50], "end": [15.50, 16.20, 11.00, 20.00]},
    "P011": {"start": [10.94, 12.76, 6.10, 7.91], "end": [20.80, 24.00, 9.30, 14.00]},
    "P012": {"start": [12.00, 12.00, 6.00, 6.00], "end": [17.30, 17.00, 11.00, 10.50]},
    "P013": {"start": [10.91, 12.03, 6.41, 5.03], "end": [20.00, 19.80, 11.00, 10.00]},
    "P014": {"start": [16.00, 12.00, 6.00, 17.00], "end": [20.20, 19.00, 12.00, 24.00]},
    "P015": {"start": [12.00, 12.00, 6.00, 6.00], "end": [17.20, 17.50, 11.00, 11.00]},
    "P016": {"start": [12.00, 15.00, 6.00, 6.00], "end": [17.06, 22.00, 11.00, 12.00]},
}


def _clean_columns(df):
    df = df.copy(); df.columns = df.columns.astype(str).str.strip(); return df


def _time_normalize(signal, n_points=101):
    signal = np.asarray(signal, dtype=float)
    if signal.ndim != 1 or len(signal) < 3:
        return np.full(n_points, np.nan)
    valid = np.isfinite(signal)
    if valid.sum() < 3:
        return np.full(n_points, np.nan)
    if not valid.all():
        idx = np.arange(len(signal), dtype=float)
        signal = np.interp(idx, idx[valid], signal[valid])
    return interp1d(np.linspace(0, 100, len(signal)), signal, kind="linear", bounds_error=False,
                    fill_value="extrapolate")(np.linspace(0, 100, n_points))


def _magnitude(df, prefix):
    cols = [f"{prefix} x", f"{prefix} y", f"{prefix} z"]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    x, y, z = [pd.to_numeric(df[c], errors="coerce").to_numpy(float) for c in cols]
    return np.sqrt(x*x + y*y + z*z)


rows, processing_log = [], []
for participant in participants:
    for trial in target_trials:
        file_name = f"{participant}-{trial:03d}.xlsx"
        if not os.path.exists(file_name):
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Missing file"})
            continue
        try:
            td = _clean_columns(pd.read_excel(file_name, sheet_name=sheet_name))
            idx = trial - 3
            if participant in cut_times:
                start_time = cut_times[participant][idx]
                start_frame, end_frame = int(round(start_time * fs)), len(td)
                end_time = end_frame / fs
            else:
                start_time = reaction_times[participant]["start"][idx]
                end_time = reaction_times[participant]["end"][idx]
                start_frame, end_frame = int(round(start_time * fs)), int(round(end_time * fs))
            start_frame, end_frame = max(0, start_frame), min(len(td), end_frame)
            if end_frame <= start_frame:
                processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Invalid cut window"})
                continue
            cropped = td.iloc[start_frame:end_frame].copy()
            pelvis = _magnitude(cropped, "Pelvis")
            left = _magnitude(cropped, "Left Foot")
            right = _magnitude(cropped, "Right Foot")
            eps = 1e-8
            rows.append({
                "Participant": participant, "Trial": trial, "Condition": trial_map[trial],
                "Start_Time_s": start_time, "End_Time_s": end_time, "Start_Frame": start_frame,
                "End_Frame": end_frame, "Segment_Samples": len(cropped),
                "Pelvis/Left Foot Redistribution": _time_normalize(pelvis / (pelvis + left + eps), n_points),
                "Pelvis/Right Foot Redistribution": _time_normalize(pelvis / (pelvis + right + eps), n_points),
            })
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Processed"})
        except Exception as exc:
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": f"{type(exc).__name__}: {exc}"})

curve_df = pd.DataFrame(rows)
processing_log_df = pd.DataFrame(processing_log)
if curve_df.empty:
    raise ValueError("No SPM trials were processed.")

participant_rows = []
for participant in participants:
    for condition in ["LOW", "HIGH"]:
        subset = curve_df[(curve_df["Participant"] == participant) & (curve_df["Condition"] == condition)]
        if len(subset) != EXPECTED_REPETITIONS:
            continue
        row = {"Participant": participant, "Condition": condition, "Valid_Trials": len(subset)}
        for outcome in spm_outcomes:
            curves = np.vstack(subset[outcome].to_numpy())
            row[outcome] = np.mean(curves, axis=0) if curves.shape == (2, n_points) and np.all(np.isfinite(curves)) else np.full(n_points, np.nan)
        participant_rows.append(row)
participant_df = pd.DataFrame(participant_rows)

cluster_rows, summary_rows = [], []
for outcome in spm_outcomes:
    low_rows, high_rows, included = [], [], []
    for participant in participants:
        low = participant_df[(participant_df["Participant"] == participant) & (participant_df["Condition"] == "LOW")]
        high = participant_df[(participant_df["Participant"] == participant) & (participant_df["Condition"] == "HIGH")]
        if len(low) == 1 and len(high) == 1:
            l = np.asarray(low.iloc[0][outcome], float); h = np.asarray(high.iloc[0][outcome], float)
            if l.shape == (n_points,) and h.shape == (n_points,) and np.all(np.isfinite(l)) and np.all(np.isfinite(h)):
                low_rows.append(l); high_rows.append(h); included.append(participant)
    if len(included) < 3:
        summary_rows.append({"Outcome": outcome, "Participants": len(included), "df": max(len(included)-1, 0),
                             "Alpha family": spm_alpha_family, "Alpha used": spm_alpha_per_outcome,
                             "Significant clusters": 0})
        continue
    low_array, high_array = np.vstack(low_rows), np.vstack(high_rows)
    spm = spm1d.stats.ttest_paired(high_array, low_array)
    ti = spm.inference(alpha=spm_alpha_per_outcome, two_tailed=True, interp=True)
    clusters = list(getattr(ti, "clusters", []))
    summary_rows.append({"Outcome": outcome, "Participants": len(included), "df": len(included)-1,
                         "Alpha family": spm_alpha_family, "Alpha used": spm_alpha_per_outcome,
                         "SPM critical threshold": float(ti.zstar), "Significant clusters": len(clusters)})
    z = np.asarray(ti.z, float)
    for k, cluster in enumerate(clusters, start=1):
        endpoints = np.asarray(cluster.endpoints, float)
        start_node, end_node = float(endpoints[0]), float(endpoints[1])
        lo, hi = max(0, int(np.floor(start_node))), min(n_points-1, int(np.ceil(end_node)))
        inds = np.arange(lo, hi+1)
        peak_idx = int(inds[np.argmax(np.abs(z[inds]))])
        peak_t = float(z[peak_idx])
        cluster_rows.append({
            "Outcome": outcome, "Cluster": k, "Start_Percent": start_node * 100/(n_points-1),
            "End_Percent": end_node * 100/(n_points-1), "Cluster p": float(cluster.P),
            "Direction": "HIGH > LOW" if peak_t > 0 else "HIGH < LOW",
            "Peak SPM t": peak_t, "Peak Percent": peak_idx * 100/(n_points-1),
            "Participants": len(included), "df": len(included)-1,
        })

spm_summary = pd.DataFrame(summary_rows)
spm_clusters = pd.DataFrame(cluster_rows)
participant_export = participant_df[["Participant", "Condition", "Valid_Trials"]].copy()
metadata = pd.DataFrame({
    "Item": ["Experimental unit", "Repeated-trial handling", "SPM test", "Multiple-comparison correction"],
    "Specification": [
        "Participant",
        "Exactly two trials are averaged within each participant-condition before SPM inference.",
        "Two-tailed participant-level paired SPM1D t-test; df = number of paired participants - 1.",
        "SPM1D controls the continuum-wise error within each trajectory; Bonferroni alpha = .05/2 = .025 is used across the two prespecified pelvis-foot trajectory outcomes.",
    ],
})
spm_path = OUTPUT_DIR / "SPM_Pelvis_Foot_Redistribution.xlsx"
with pd.ExcelWriter(spm_path, engine="openpyxl") as writer:
    spm_summary.to_excel(writer, sheet_name="SPM Summary", index=False)
    spm_clusters.to_excel(writer, sheet_name="Significant Clusters", index=False)
    participant_export.to_excel(writer, sheet_name="Participant Means", index=False)
    curve_df.drop(columns=spm_outcomes).to_excel(writer, sheet_name="Trial Processing", index=False)
    processing_log_df.to_excel(writer, sheet_name="Processing Log", index=False)
    metadata.to_excel(writer, sheet_name="Analysis Metadata", index=False)
_format_workbook(spm_path)


## Table: EMG Redistribution

In [ ]:
# =====================================================
# LOAD DATASET
# =====================================================
data = pd.read_excel(
    "EMG Data Analysis.xlsx",
    sheet_name="TransientState",
).copy()

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

# Restrict analysis to Area 1.
data = data[
    data["Area"] == 1
].copy()


# =====================================================
# CONDITION MAPPING
# =====================================================
condition_map = {
    "WRLS01": "LOW",
    "WRLS02": "LOW",
    "WRHS01": "HIGH",
    "WRHS02": "HIGH",
}

data["Group"] = data[
    "Condition"
].map(condition_map)

data = data[
    data["Group"].isin(
        ["LOW", "HIGH"]
    )
].copy()


# =====================================================
# MUSCLE GROUPS
# =====================================================
right_proximal = ["RRF"]
right_distal = ["RTA", "RGAL"]

left_proximal = ["LRF"]
left_distal = ["LTA", "LGAL"]


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE REDISTRIBUTION RATIO PER ORIGINAL TRIAL
# =====================================================
# Each Condition code represents one original trial:
# WRLS01 and WRLS02 = two LOW trials
# WRHS01 and WRHS02 = two HIGH trials

trial_rows = []

group_columns = [
    "Participant",
    "Condition",
    "Group",
]

for keys, trial_data in data.groupby(
    group_columns
):

    participant = keys[0]
    condition = keys[1]
    group = keys[2]

    # Require all specified sensors to be represented.
    available_sensors = set(
        trial_data["sensor"].dropna()
    )

    required_sensors = set(
        right_proximal
        + right_distal
        + left_proximal
        + left_distal
    )

    if not required_sensors.issubset(
        available_sensors
    ):
        continue

    right_proximal_sum = trial_data[
        trial_data["sensor"].isin(
            right_proximal
        )
    ]["iEMG"].sum(min_count=1)

    right_distal_sum = trial_data[
        trial_data["sensor"].isin(
            right_distal
        )
    ]["iEMG"].sum(min_count=1)

    left_proximal_sum = trial_data[
        trial_data["sensor"].isin(
            left_proximal
        )
    ]["iEMG"].sum(min_count=1)

    left_distal_sum = trial_data[
        trial_data["sensor"].isin(
            left_distal
        )
    ]["iEMG"].sum(min_count=1)

    right_denominator = (
        right_proximal_sum
        + right_distal_sum
    )

    left_denominator = (
        left_proximal_sum
        + left_distal_sum
    )

    if (
        not np.isfinite(right_denominator)
        or not np.isfinite(left_denominator)
        or right_denominator == 0
        or left_denominator == 0
    ):
        continue

    right_ratio = (
        right_proximal_sum
        / right_denominator
    )

    left_ratio = (
        left_proximal_sum
        / left_denominator
    )

    trial_rows.append({
        "Participant": participant,
        "Condition": condition,
        "Group": group,
        "RR_Right": right_ratio,
        "RR_Left": left_ratio,
    })


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No EMG redistribution ratios were computed. "
        "Check the input workbook and sensor labels."
    )


emgr_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Group"],
    value_vars=["RR_Right", "RR_Left"], var_name="Metric", value_name="Value",
)
emgr_long["Metric"] = emgr_long["Metric"].map({"RR_Right": "Right EMG Redistribution", "RR_Left": "Left EMG Redistribution"})
run_lmm_family(emgr_long, "Metric", "Value", "Participant", "Group", ["LOW", "HIGH"],
               "Table_EMG_Redistribution.xlsx", decimals=3)


## Table: Margin of Stability (MoS)

In [ ]:
fs = 60
g = 9.81

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES
# =====================================================
cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(ddof=1)
    se_value = sd_value / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean_value - t_critical * se_value
    ci_high = mean_value + t_critical * se_value

    return (
        f"{mean_value:.3f} ± {sd_value:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL ML MARGIN OF STABILITY
# =====================================================
trial_rows = []

required_com_columns = [
    "CoM pos y",
    "CoM vel y",
    "CoM pos z",
]

required_position_columns = [
    "Left Foot y",
    "Right Foot y",
]

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                com = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                segment_position = pd.read_excel(
                    file_name,
                    sheet_name="Segment Position",
                )

                com.columns = (
                    com.columns
                    .astype(str)
                    .str.strip()
                )

                segment_position.columns = (
                    segment_position.columns
                    .astype(str)
                    .str.strip()
                )

                if any(
                    column not in com.columns
                    for column in required_com_columns
                ):
                    continue

                if any(
                    column not in segment_position.columns
                    for column in required_position_columns
                ):
                    continue

                timing_index = trial - 3

                if participant in cut_times:
                    start_frame = int(round(
                        cut_times[participant][timing_index]
                        * fs
                    ))
                    end_frame = len(com)

                else:
                    start_frame = int(round(
                        reaction_times[participant]["start"][
                            timing_index
                        ] * fs
                    ))

                    end_frame = int(round(
                        reaction_times[participant]["end"][
                            timing_index
                        ] * fs
                    ))

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    end_frame,
                    len(com),
                    len(segment_position),
                )

                if end_frame <= start_frame:
                    continue

                com_y = (
                    com["CoM pos y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                com_velocity_y = (
                    com["CoM vel y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                com_z = (
                    com["CoM pos z"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                left_foot_y = (
                    segment_position["Left Foot y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                right_foot_y = (
                    segment_position["Right Foot y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                valid_signal = (
                    np.isfinite(com_y)
                    & np.isfinite(com_velocity_y)
                    & np.isfinite(com_z)
                    & np.isfinite(left_foot_y)
                    & np.isfinite(right_foot_y)
                )

                if not np.any(valid_signal):
                    continue

                com_y = com_y[valid_signal]
                com_velocity_y = (
                    com_velocity_y[valid_signal]
                )
                com_z = com_z[valid_signal]
                left_foot_y = left_foot_y[valid_signal]
                right_foot_y = right_foot_y[valid_signal]

                pendulum_length = np.mean(com_z)

                if (
                    not np.isfinite(pendulum_length)
                    or pendulum_length <= 0
                ):
                    continue

                omega_0 = np.sqrt(
                    g / pendulum_length
                )

                xcom_ml = (
                    com_y
                    + com_velocity_y / omega_0
                )

                bos_ml = np.maximum(
                    left_foot_y,
                    right_foot_y,
                )

                mos_ml = bos_ml - xcom_ml
                mos_ml = mos_ml[
                    np.isfinite(mos_ml)
                ]

                if len(mos_ml) == 0:
                    continue

                trial_rows.append({
                    "Participant": participant,
                    "Condition": condition,
                    "Trial": trial,
                    "Mean_ML_MoS": np.mean(mos_ml),
                    "Min_ML_MoS": np.min(mos_ml),
                    "Max_ML_MoS": np.max(mos_ml),
                })

            except Exception:
                continue


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No MoS results were computed. Check the working "
        "directory and input workbook structure."
    )


mos_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["Mean_ML_MoS", "Min_ML_MoS", "Max_ML_MoS"], var_name="Metric", value_name="Value",
)
mos_long["Metric"] = mos_long["Metric"].map({"Mean_ML_MoS": "Mean MoS", "Min_ML_MoS": "Minimum MoS", "Max_ML_MoS": "Maximum MoS"})
run_lmm_family(mos_long, "Metric", "Value", "Participant", "Condition", ["LOW", "HIGH"],
               "Table_Margin_of_Stability.xlsx", decimals=3)


## Table: Steady-State Kinematics

In [ ]:
input_path = "Xsens Data Analysis.xlsx"
sheet_name = "Sheet1"
area_value = 1
subject_col = "Participant"
trial_col = "Condition"
condition_order = ["No Robot", "Low Speed", "High Speed"]
condition_map = {
    "NR01": "No Robot", "NR02": "No Robot",
    "WRLS01": "Low Speed", "WRLS02": "Low Speed",
    "WRHS01": "High Speed", "WRHS02": "High Speed",
}

data = pd.read_excel(input_path, sheet_name=sheet_name).copy()
data.columns = data.columns.astype(str).str.strip()
required = [subject_col, trial_col, "Area"]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
data = data[data["Area"] == area_value].copy()
data["Group"] = data[trial_col].map(condition_map)
data = data[data["Group"].isin(condition_order)].copy()

data["CoM_pos_mag"] = np.sqrt(data["CoM pos x"]**2 + data["CoM pos y"]**2 + data["CoM pos z"]**2)
data["CoM_vel_mag"] = np.sqrt(data["CoM vel x"]**2 + data["CoM vel y"]**2 + data["CoM vel z"]**2)
data["CoM_acc_mag"] = np.sqrt(data["CoM acc x"]**2 + data["CoM acc y"]**2 + data["CoM acc z"]**2)

kinematic_columns = [
    "CoM_pos_mag", "CoM_vel_mag", "CoM_acc_mag",
    "Right Shoulder Flexion/Extension", "Right Shoulder Internal/External Rotation", "Right Shoulder Abduction/Adduction",
    "Right Knee Flexion/Extension", "Right Knee Internal/External Rotation", "Right Knee Abduction/Adduction",
    "Left Knee Flexion/Extension", "Left Knee Internal/External Rotation", "Left Knee Abduction/Adduction",
]
rename = {
    "CoM_pos_mag": "CoM Position Magnitude", "CoM_vel_mag": "CoM Velocity Magnitude", "CoM_acc_mag": "CoM Acceleration Magnitude",
}
kin_rows = []
for variable in kinematic_columns:
    if variable not in data.columns:
        continue
    t = (data.groupby([subject_col, trial_col, "Group"], as_index=False)[variable].mean()
         .rename(columns={variable: "Value"}))
    t["Metric"] = rename.get(variable, variable)
    kin_rows.append(t[["Metric", subject_col, trial_col, "Group", "Value"]])
kin_long = pd.concat(kin_rows, ignore_index=True)
run_lmm_family(kin_long, "Metric", "Value", subject_col, "Group", condition_order,
               "Table_Kinematics.xlsx", decimals=2)


## Table: Steady-State EMG

In [ ]:
input_path = "EMG Data Analysis.xlsx"
sheet_name = "SteadyState"
subject_col = "Participant"
condition_col = "Condition"
sensor_col = "sensor"
value_col = "iEMG"
area_col = "Area"
condition_order = ["NR", "LOW", "HIGH"]
condition_map = {
    "NR01": "NR", "NR02": "NR", "WRLS01": "LOW", "WRLS02": "LOW", "WRHS01": "HIGH", "WRHS02": "HIGH",
}
sensors_to_analyze = ["LGAL", "LRF", "LTA", "LUT", "RBB", "RGAL", "RRF", "RTA", "RTB", "RUT"]

data = pd.read_excel(input_path, sheet_name=sheet_name).copy()
data.columns = data.columns.astype(str).str.strip()
required = [subject_col, condition_col, sensor_col, value_col]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if area_col in data.columns:
    data = data[data[area_col] == 1].copy()
data[subject_col] = data[subject_col].astype(str).str.strip()
data[condition_col] = data[condition_col].astype(str).str.strip()
data[sensor_col] = data[sensor_col].astype(str).str.strip().str.upper()
data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
data["Group"] = data[condition_col].map(condition_map)
data = data[data["Group"].isin(condition_order) & data[sensor_col].isin(sensors_to_analyze)].dropna(subset=[subject_col, condition_col, sensor_col, value_col, "Group"]).copy()
trial_level = (data.groupby([subject_col, sensor_col, condition_col, "Group"], as_index=False)[value_col].mean())
trial_level = trial_level.rename(columns={sensor_col: "Metric", value_col: "Value"})
run_lmm_family(trial_level, "Metric", "Value", subject_col, "Group", condition_order,
               "Table_EMG_iEMG.xlsx", decimals=2)
